# Zarr: A cloud-optimized format and protocol for "virtual" datacubes

Francesco Nattino, 2026-04-09

## Introduction

### Zarr, in a nutshell

* an **open-source** file format.
* for **chunked** and **compressed** multi-dimensional arrays.
* designed for large datasets, **cloud-native**.

A few concepts in Zarr (from the [Zarr specification](https://zarr-specs.readthedocs.io/en/latest/v3/core/index.html#concepts-and-terminology), version 3):

<div>
<img src="./fig/terminology-hierarchy.excalidraw.png" width="70%"/>
</div>

### Cloud-native

-   **Consolidated metadata**: structural metadata in one read.
-   **Chunked storage**: read only the data you need.
-   **Multiple files**: parallel read/write, scalable.
-   Natively works with **object storage**.

### Where is it used?

- Climate data and Earth system models.
- Earth observation (satellite images).
- Also non-geospatial: bioimaging, electron microscopy.

## Zarr in practice

### Implementations

Open-source [specification](https://zarr-specs.readthedocs.io/en/latest/specs.html), with [implementations](https://zarr.dev/implementations/) in:

-   Python
-   C++
-   Javascript
-   Julia
-   Rust
-   ...

### Zarr-Python

[Zarr-Python](https://zarr.readthedocs.io) is the reference Zarr implementation, with Numpy- and h5py-like interface:

In [ ]:
import zarr
root = zarr.group(store="data/example-1.zarr", overwrite=True)
x = root.create_array("x", shape=(100, 200), chunks=(50, 50), dtype="f8")
y = root.create_array("y", shape=(100, 200), chunks=(50, 50), dtype="f8")

In [ ]:
! tree data/example-1.zarr

In [ ]:
import numpy as np
x[:, :] = np.random.random((100, 200))

In [ ]:
! tree data/example-1.zarr

The store can be essentially anything dict-like, even a dictionary:

In [ ]:
store = {}
root = zarr.group(store=store)
x = root.create_array("x", shape=(100, 200), chunks=(50, 50), dtype="f8")
y = root.create_array("y", shape=(100, 200), chunks=(50, 50), dtype="f8")

In [ ]:
x[:, :] = np.random.random((100, 200))

In [ ]:
store

### Xarray

[Xarray](https://docs.xarray.dev) has good support to read from/write to Zarr stores, also with [Dask](https://docs.dask.org): 

In [ ]:
import dask.array as da
import xarray as xr

x = xr.DataArray(
    name="x",
    data=da.random.random((25, 512, 512), chunks=(25, 256, 256)),
    coords={
        "time": range(2000, 2025),
        "lat": np.linspace(-90, 90, 512),
        "lon": np.linspace(-180, 180, 512)
    },
    dims=("time", "lat", "lon"),
)

x

In [ ]:
x.to_zarr("data/example-2.zarr", mode="w", consolidated=True, zarr_format=3)

In [ ]:
! tree data/example-2.zarr

In [ ]:
! cat data/example-2.zarr/zarr.json

An example of "real" Zarr dataset is a version of [ERA5](https://www.ecmwf.int/en/forecasts/dataset/ecmwf-reanalysis-v5) which is hosted as a public dataset in [a Google cloud bucket](https://github.com/google-research/arco-era5): 

In [ ]:
ds = xr.open_zarr(
    'gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3',
    chunks=None,
    storage_options=dict(token='anon'),
)
ds

In [ ]:
ar_full_37_1h = ds.sel(time=slice(ds.attrs['valid_time_start'], ds.attrs['valid_time_stop']))
ar_full_37_1h["soil_temperature_level_1"].isel(time=0).plot.imshow()

## Virtual Zarr stores

Virtual Zarr stores can be created to provide Zarr-like access to (collections of) netCDF, TIFF, GRIB2, ... without duplicating the data. 

Virtual Zarrs can be created with [kerchunk](https://fsspec.github.io/kerchunk/) or [VirtualiZarr](https://virtualizarr.readthedocs.io) - we'll show an example usage of the former in the following.

### With NetCDF/HDF5 (locally)

We consider a set of NetCDF files from the [Daymet dataset (v4)](https://doi.org/10.3334/ORNLDAAC/2129), which includes daily surface weather variables for North America (here we consider only the Hawaii region).

In [ ]:
import glob
hdf_files = glob.glob("data/daymet-daily-v4/region-hi/*.nc")
hdf_files

In [ ]:
ds = xr.open_dataset(hdf_files[0], chunks={})
ds

We extract references to the individual chunks within one of the files:

In [ ]:
from kerchunk.hdf import SingleHdf5ToZarr

chunks = SingleHdf5ToZarr(hdf_files[0], inline_threshold=300)
refs = chunks.translate()

In [ ]:
import pprint
pprint.pprint(refs)

Now for all the NetCDFs, saving the references as JSON files:

In [ ]:
import json
import pathlib

# Extract references for all files
for hdf_file in hdf_files:
    chunks = SingleHdf5ToZarr(hdf_files[0], inline_threshold=300)
    references = chunks.translate()
    json_file = pathlib.Path(hdf_file).with_suffix(".json")
    with json_file.open("w") as f:
        f.write(json.dumps(references))

In [ ]:
json_files = glob.glob("data/daymet-daily-v4/region-hi/*.json")
json_files

All these reference files can be combined in a single file, which can be used to describe the full collection:

In [ ]:
# needed to run the following cell in Jupyter
import nest_asyncio
nest_asyncio.apply()

In [ ]:
from kerchunk.combine import MultiZarrToZarr

chunks = MultiZarrToZarr(
    json_files,
    concat_dims=["time"],
    identical_dims=["x", "y", "nv"],
)
references = chunks.translate()

with open("data/daymet-daily-v4/region-hi.json", "wb") as f:
    f.write(json.dumps(references).encode())

The combined reference file can now be opened as a virtual datacube with Xarray, using Zarr-Python - no need for h5py/netCDF4/h5netcdf:

In [ ]:
ds = xr.open_dataset(
    "data/daymet-daily-v4/region-hi.json", 
    engine="kerchunk",
    chunks={}
)
ds

In [ ]:
ds["dayl"].isel(time=0).plot.imshow()

### With NetCDF/HDF5 (on SURF dCache)

We have run the same procedure (extracting chunk references, combining them in a single reference file) for the same [Daymet dataset (v4)](https://doi.org/10.3334/ORNLDAAC/2129), but considering the full North American continent, 3 variables and 40 years of data. 

We have downloaded (part of) this dataset to the [SURF dCache storage system](https://www.surf.nl/en/services/storage-data-management/dcache), which we access with the [fsspec](https://filesystem-spec.readthedocs.io)-compatible interface that we have developed for dCache ([dcachefs](https://github.com/RS-DAT/dcachefs)). NOTE: accessing dCache requires authentication, which we have set up via the configuration framework of fsspec (here, with a config file in `~/.config/fsspec`).

[This script](./scripts/kerchunk-daymet-v4.py) creates the following combined reference file:

In [ ]:
! du -h data/daymet-daily-v4/region-na.json 

Opening the combined reference file, shows the 120 NetCDF files as one virtual datacube:

In [ ]:
import dcachefs  # register dcachefs as fsspec known implementation 

ds = xr.open_dataset(
    "data/daymet-daily-v4/region-na.json", 
    engine="kerchunk", 
    chunks={}
)

In [ ]:
ds

In [ ]:
ds["tmax"].isel(time=0).plot.imshow()